## Scenario 3: Multiple data scientists working on multiple ML models

MLflow setup:
* Tracking server: yes, remote server (EC2).
* Backend store: postgresql database.
* Artifacts store: s3 bucket.

The experiments can be explored by accessing the remote server.

The example uses AWS to host a remote server. In order to run the example you'll need an AWS account. Follow the steps described in the file `mlflow_on_aws.md` to create a new AWS account and launch the tracking server. 

In [1]:
import mlflow
import os

#os.environ["AWS_PROFILE"] = "" # fill in with your AWS profile. More info: https://docs.aws.amazon.com/sdk-for-java/latest/developer-guide/setup.html#setup-credentials

TRACKING_SERVER_HOST = "18.219.33.131" # fill in with the public DNS of the EC2 instance
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://18.219.33.131:5000'


In [3]:
mlflow.search_experiments() # list_experiments API has been removed, you can use search_experiments instead.()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-32/1', creation_time=1793300392169, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793300392169, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-32/0', creation_time=1793299268447, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1793299268447, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2026/10/29 19:09:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/29 19:09:31 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


default artifacts URI: 's3://mlflow-artifacts-remote-32/1/a9ad1b37a6e144da9d1f22b00a9ddbe7/artifacts'
🏃 View run worried-stag-203 at: http://18.219.33.131:5000/#/experiments/1/runs/a9ad1b37a6e144da9d1f22b00a9ddbe7
🧪 View experiment at: http://18.219.33.131:5000/#/experiments/1


In [6]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlflow-artifacts-remote-32/1', creation_time=1793300392169, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793300392169, lifecycle_stage='active', name='my-experiment-1', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='s3://mlflow-artifacts-remote-32/0', creation_time=1793299268447, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1793299268447, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient


client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

In [8]:
client.search_registered_models()

[]

In [9]:
run_id = client.search_runs(experiment_ids=['1'])[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2026/10/29 19:10:23 WARNING mlflow.tracking._model_registry.fluent: Run with id a9ad1b37a6e144da9d1f22b00a9ddbe7 has no artifacts at artifact path 'models', registering model based on models:/m-fc51081c48f7403a9ae2a80787d35ab4 instead
2026/10/29 19:10:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1793301023841, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1793301023841, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='a9ad1b37a6e144da9d1f22b00a9ddbe7', run_link='', source='models:/m-fc51081c48f7403a9ae2a80787d35ab4', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>